# IRE Search Engine — Interview Study Guide (full)

One place to **learn the project**, **practice what to say**, and **see the architecture visually**.

**Diagrams:** Mermaid renders in GitHub, VS Code Markdown preview, or [mermaid.live](https://mermaid.live). Jupyter may need an extension for Mermaid.

**Run code cells** from the repo root after `pip install -e .` (add `pip install -e ".[ui]"` for semantic tests). If the notebook lives in `notebooks/`, the snippets add `../src` to `sys.path` automatically.


## Table of contents

1. [How to run this project](#How-to-run-this-project)
2. [What features this includes](#What-features-this-includes)
3. [Why this project is useful](#Why-this-project-is-useful)
4. [Two tracks: lexical vs semantic](#Two-tracks-lexical-vs-semantic)
5. [How to explain it in an interview](#How-to-explain-it-in-an-interview)
6. [Visual overview (diagrams)](#Visual-overview-diagrams)
7. [Executive summary](#Executive-summary)
8. [Repository layout](#Repository-layout)
9. [Configuration and index names](#Configuration-and-index-names)
10. [End-to-end data flow](#End-to-end-data-flow)
11. [Strategy pattern (core design)](#Strategy-pattern-core-design)
12. [Scoring: Boolean through BM25](#Scoring-Boolean-through-BM25)
13. [Preprocessing: stop words and stemming](#Preprocessing-stop-words-and-stemming)
14. [TF-IDF and BM25 (simple math)](#TF-IDF-and-BM25-simple-math)
15. [TAAT vs DAAT](#TAAT-vs-DAAT)
16. [Storage and compression](#Storage-and-compression)
17. [Query processing](#Query-processing)
18. [Evaluation and MetricsCollector](#Evaluation-and-MetricsCollector)
19. [Local semantic search (Chroma + Streamlit)](#Local-semantic-search-Chroma--Streamlit)
20. [Code map (where to look)](#Code-map-where-to-look)
21. [Commands cheat sheet](#Commands-cheat-sheet)
22. [STAR story template](#STAR-story-template)
23. [Interview Q and A](#Interview-Q-and-A)
24. [This project vs industry tools](#This-project-vs-industry-tools)
25. [Trade-offs](#Trade-offs)
26. [Honest limitations](#Honest-limitations)
27. [One-page cheat sheet](#One-page-cheat-sheet)


## How to run this project

**1. Open a terminal in the project root** (the folder that contains `cli/`, `src/`, `ui.py`).

**2. Create and activate a virtual environment (recommended)**

```text
python -m venv env
# Windows
env\Scripts\activate
# macOS / Linux
source env/bin/activate
```

**3. Install**

```text
pip install -U pip
pip install -e .
pip install -r requirements.txt
```

**4. Lexical search (inverted index, BM25, etc.)**

```text
python cli/build.py -x 4 -y 1 -z 1 -optim 0 --limit 100
python cli/query.py -x 4 -y 1 -z 1 -q T -optim 0 --query "machine learning"
```

**5. Tests**

```text
python -m pytest tests/ -q
```

**6. Semantic UI (optional, large install)**

```text
pip install -e ".[ui]"
streamlit run ui.py
```

Paste a **folder path** in the sidebar, click **Index now**, then search. Data is stored under `.chroma_db/`. See `README.md` and `docs/SEMANTIC_SEARCH.md` for details.


## What features this includes

| Area | What you get |
|------|----------------|
| **Lexical IR** | Inverted index, **Boolean** (AND/OR/NOT, phrases), **TF**, **TF-IDF**, **BM25** |
| **Storage** | JSON or SQLite on disk; optional **Elias** or **zlib** compression on postings |
| **Query modes** | **TAAT** and **DAAT** for ranked models (same index file, runtime choice) |
| **Text prep** | Lowercase, tokenize, **English stop words**, **Porter stemming** (NLTK) |
| **Evaluation** | `cli/evaluate.py` + **MetricsCollector**: latency, throughput, memory → JSON |
| **Semantic (optional)** | **ChromaDB** + **Sentence-Transformers** (`all-MiniLM-L6-v2`), **FileCrawler** for PDF/DOCX/PPTX/MD, **Streamlit** `ui.py` |

```mermaid
flowchart TB
  R[IRE Search Engine]
  R --> L[Lexical: SelfIndexer + 4 scorers + JSON/SQLite + compression]
  R --> S[Semantic: LocalIndexer + Chroma + Streamlit]
  R --> T[Quality: pytest + benchmarks]
```


## Why this project is useful

| Audience | How you use it |
|----------|----------------|
| **You (learning)** | See how **postings**, **IDF**, **BM25**, and **TAAT/DAAT** connect in real code—not only slides. |
| **Interviews** | Tell a clear story: **Strategy pattern**, **trade-offs** (speed vs relevance, disk vs CPU), **evaluation** methodology. |
| **Portfolio** | Demonstrates **two paradigms**: classic IR **and** embedding search (local, no cloud DB). |
| **Personal use** | Semantic UI can **search your own folder** of notes and PDFs (offline, after install). |

**Simple sentence for interviewers:** *“It is a pluggable search engine: one indexer class, swappable ranking and storage, plus an optional local vector search path for semantic similarity.”*


## Two tracks: lexical vs semantic

Think of the project as **two** ways to search—**keyword/statistics** vs **meaning vectors**.

```mermaid
flowchart TB
    subgraph lex[Track A — Lexical IR]
        L1[Documents + preprocess]
        L2[Inverted index postings]
        L3[BM25 / TF-IDF / Boolean]
        L4[indices folder JSON or SQLite]
    end
    subgraph sem[Track B — Semantic optional]
        S1[Files PDF DOCX PPTX MD]
        S2[FileCrawler text]
        S3[Embeddings MiniLM]
        S4[Chroma .chroma_db]
        S5[Streamlit ui.py]
    end
    L1 --> L2 --> L3 --> L4
    S1 --> S2 --> S3 --> S4
    S5 --> S2
    S5 --> S3
```

- **Track A** is the course-style **IR engine** (reproducible configs, benchmarks).
- **Track B** is **local semantic** search over **your files**—good for demos and “how is this different from keyword search?”


## How to explain it in an interview

### 30-second pitch (read aloud)

> I built a **search engine in Python** over a large text collection. The core is an **inverted index** with **pluggable ranking**: Boolean through **BM25**, and **JSON or SQLite** storage with optional **compression**. I can **benchmark** latency and throughput. I also added an **optional local semantic** path: **embeddings** in **ChromaDB** and a **Streamlit** UI to search folders of PDFs and Office files—so I can contrast **keyword** vs **meaning-based** retrieval.

### 2 minutes (structure)

1. **Goal** — Search and evaluate over **100K-style** mixed documents with **named configs**.
2. **Design** — **One `SelfIndexer`**, not 24 classes: **Strategy pattern** (`ScoringStrategy` + `StorageBackend`).
3. **Technical** — Postings, metadata (IDF, lengths), preprocessing, **TAAT/DAAT**.
4. **Extra** — **Local semantic** search with **Chroma** + **Sentence-Transformers** (no Elasticsearch).
5. **Scope** — **Single machine**, educational clarity; **not** a managed cloud product.

### 5 minutes (if they want depth)

Walk the **data flow** diagram, then **one trade-off** (e.g. BM25 vs Boolean speed), then **how you would extend** (new scorer = implement interface + factory).

### If they say “walk me through the code”

`create_indexer` → `SelfIndexer.create_index` → `build_postings` → `save` → `load_index` → `score_query`. Point to **`core/indexer.py`** and **`scoring/strategies.py`**. Mention **`integrations/chroma_local`** only if they ask about semantic search.


## Visual overview diagrams

### System context

```mermaid
flowchart TB
    subgraph data[Data]
        W[Wiki parquet]
        N[News zips]
        F[User files PDF MD]
    end
    subgraph ire[ire_search package]
        DL[io data_loader]
        FC[file_crawler]
        PP[preprocessor]
        IX[SelfIndexer]
        SC[scoring strategies]
        ST[storage backends]
        LI[LocalIndexer Chroma]
    end
    W --> DL
    N --> DL
    F --> FC
    DL --> PP
    FC --> LI
    PP --> IX
    IX --> SC
    IX --> ST
    ST --> IDX[(indices)]
    LI --> CH[(.chroma_db)]
```

### Strategy pattern (composition)

```mermaid
flowchart LR
    F[create_indexer x y z] --> SI[SelfIndexer]
    SI --> G[get_scorer]
    SI --> H[get_storage]
    G --> B1[Boolean TF TFIDF BM25]
    H --> B2[JSON SQLite]
```

### Build vs query (lexical)

```mermaid
flowchart LR
    subgraph B[Build]
        D[Docs] --> P[Preprocess]
        P --> Po[Postings]
        Po --> M[Metadata]
        M --> S[Save]
    end
    subgraph Q[Query]
        L[Load] --> SQ[score_query]
    end
    S -.-> L
```


## Executive summary

| Piece | What it is |
|-------|------------|
| **Corpus** | Up to ~100K docs (wiki + news) for experiments; **semantic** path uses **your folder** |
| **Lexical index** | **`SelfIndexer`** + **scorer** + **storage** |
| **Scorers** | x=1 Boolean, 2 TF, 3 TF-IDF, 4 BM25 |
| **Storage** | y=1 JSON, y=2 SQLite; z=compression |
| **Eval** | Latency, QPS, memory; **MetricsCollector** in `cli/evaluate.py` |
| **Semantic** | **Chroma** + **MiniLM** + **Streamlit** |

**Identifier:** `SelfIndex_i{x}d{y}c{z}o{optim}` — query mode **TAAT/DAAT is not** in the filename (runtime choice).


## Repository layout

```text
IRE_Assignment1/
├── ui.py                 # Streamlit semantic UI
├── main.py               # Optional entry to cli/build|query|evaluate
├── cli/                  # build, query, evaluate, plots
├── src/ire_search/
│   ├── core/             # SelfIndexer, SearchEngine, preprocessor, file_crawler
│   ├── scoring/          # strategies.py
│   ├── storage/
│   ├── compression/
│   ├── io/
│   ├── evaluation/
│   └── integrations/chroma_local/   # LocalIndexer
├── tests/
├── docs/
├── notebooks/            # this guide
└── indices/              # lexical index files
```

**Imports:** `from ire_search import create_indexer` — package lives under `src/ire_search/` with `pip install -e .`.


## Configuration and index names

| Flag | Meaning |
|------|---------|
| **-x** | 1 Boolean, 2 TF, 3 TF-IDF, 4 BM25 |
| **-y** | 1 JSON, 2 SQLite |
| **-z** | 1 none, 2 Elias, 3 zlib |
| **-optim** | optimization flags (see `USER_GUIDE`) |
| **-q T/D** | TAAT vs DAAT for ranked models |

**Remember:** TAAT/DAAT does **not** change the on-disk index name—only how you **traverse** postings at query time.


## End-to-end data flow

1. **Load** documents (Parquet/ZIP or your own dicts).
2. **Preprocess** text (or pass `tokens`).
3. **build_postings** per document per scorer.
4. **compute_metadata** (IDF, norms, lengths).
5. **save** via storage backend (maybe compress).
6. **load** at query time.
7. **score_query** with TAAT or DAAT (or Boolean parser).

```mermaid
sequenceDiagram
    participant U as User
    participant I as SelfIndexer
    participant S as Scorer
    U->>I: query(text)
    I->>S: score_query
    S-->>I: ranked doc_ids
    I-->>U: results
```


## Strategy pattern (core design)

**Why it matters:** You can swap **how you rank** and **where you save** without rewriting the whole indexer.

**Interview line:** *“I used the Strategy pattern so `SelfIndexer` stays one class; I inject a `ScoringStrategy` and a `StorageBackend` from factories.”*


## Scoring: Boolean through BM25

| x | Name | Idea in one line |
|---|------|------------------|
| 1 | Boolean | Sets of doc ids; AND/OR/NOT; phrase uses positions |
| 2 | TF | Term frequency normalized by doc length |
| 3 | TF-IDF | TF × IDF; rare terms get higher IDF |
| 4 | BM25 | Industry standard; length-aware TF saturation |

**Interview:** *“BM25 is usually the best default for relevance among these; Boolean is for structured filters.”*

```mermaid
flowchart LR
    subgraph x1[x equals 1]
        A[Boolean] --> P1[positions plus sets]
    end
    subgraph x4[x equals 4]
        B[BM25] --> P4[IDF plus lengths]
    end
```


## Preprocessing: stop words and stemming

**Where:** `core/preprocessor.py` (when you use `content`, not pre-tokenized `tokens`).

**Pipeline:** lowercase → strip URLs/HTML → tokenize → **remove English stop words** → **Porter stem**.

**Interview:** *“Stop words reduce noise for ranking; stemming maps `running` and `run` to the same stem. If you pass `tokens` yourself, you skip this pipeline.”*


## TF-IDF and BM25 (simple math)

**TF-IDF:** rare terms (low document frequency) get **higher IDF**, so they push ranking more than common words.

**BM25:** adds **length normalization** and **term saturation** (extra repeats of a word help less and less).

**You do not need to derive formulas on a whiteboard**—say what each term *means* and that **metadata** stores IDF and lengths.


## TAAT vs DAAT

**TAAT:** loop **query terms** first; accumulate scores in a hash map.

**DAAT:** walk posting lists in **document id order** with synchronized pointers.

**Same index file** for both. **Interview:** *“It is a query-time algorithm choice, so it is not in the index filename.”*

```mermaid
flowchart TB
    subgraph T[TAAT]
        t1[term1 list] --> acc[accumulator]
        t2[term2 list] --> acc
    end
    subgraph D[DAAT]
        d1[sync pointers] --> merge[merge by doc id]
    end
```


## Storage and compression

**JSON:** simple, load whole file.

**SQLite:** better for large lexicons on disk.

**Compression:** applied **inside** storage when saving postings; trade **CPU** for **disk**.

**Interview:** *“The scorer does not know about compression; the storage layer handles it.”*


## Query processing

- **Ranked:** TAAT or DAAT inside `score_query`.
- **Boolean:** shunting-yard to RPN, then set operations on posting lists.

See `docs/QUERY_PROCESSING.md` for detail.


## Evaluation and MetricsCollector

**`cli/evaluate.py`** loads a built `SelfIndex_*` index, runs a query file, records **latency percentiles**, **QPS**, **disk and RAM**.

**`MetricsCollector`** wraps each query with timing and memory traces; results include **`collector_aggregates`** in JSON.

**Interview:** *“I measured latency, throughput, and memory with a scripted benchmark—not a toy loop.”*


## Local semantic search (Chroma + Streamlit)

**What it is:** **Embeddings** (vectors of meaning) stored in **ChromaDB** under `.chroma_db/`, model **`all-MiniLM-L6-v2`**.

**`file_crawler`:** walks a folder recursively; extracts **PDF** (pymupdf4llm), **DOCX/PPTX/MD** (markitdown).

**`ui.py`:** Streamlit **folder path** → **Index now** → **search** with **similarity score**; **Reset** clears the collection.

**Why mention it:** Interviewers often ask **keyword vs semantic**—this is your **semantic** side without running a cloud service.

```mermaid
sequenceDiagram
    participant U as User
    participant UI as Streamlit ui.py
    participant FC as FileCrawler
    participant LI as LocalIndexer
    participant CH as Chroma
    U->>UI: folder path
    UI->>FC: extract text
    FC->>LI: documents
    LI->>CH: embeddings upsert
    U->>UI: query
    UI->>LI: search
    LI->>CH: query embedding
    CH-->>LI: nearest neighbors
    LI-->>UI: scores plus snippets
```


## Code map (where to look)

| Topic | File |
|-------|------|
| Indexing + query dispatch | `core/indexer.py` |
| Boolean TF TFIDF BM25 | `scoring/strategies.py` |
| JSON SQLite compression | `storage/backends.py` |
| Preprocess | `core/preprocessor.py` |
| Load wiki/news | `io/data_loader.py` |
| Benchmarks | `cli/evaluate.py`, `evaluation/metrics_collector.py` |
| Semantic | `integrations/chroma_local/local_indexer.py`, `core/file_crawler.py`, `ui.py` |


## Commands cheat sheet

| Goal | Command |
|------|---------|
| Install | `pip install -e .` |
| Build BM25 small | `python cli/build.py -x 4 -y 1 -z 1 -optim 0 --limit 100` |
| Query | `python cli/query.py -x 4 -y 1 -z 1 -q T -optim 0 --query "..."` |
| Evaluate | `python cli/evaluate.py -x 4 -y 1 -z 1 -optim 0` |
| Tests | `python -m pytest tests/ -q` |
| Semantic UI | `pip install -e ".[ui]"` then `streamlit run ui.py` |


## STAR story template

**S** — Course or self-directed project to build a **full IR pipeline** with **benchmarks** and **pluggable** design.

**T** — Implement **indexing**, **multiple rankers**, **storage**, **compression**, and **evaluation**; add **optional semantic** search.

**A** — **Strategy pattern** for scorer + storage; **TAAT/DAAT**; **MetricsCollector**; **Chroma** + **Streamlit** for a second demo track.

**R** — **Named configs** (`SelfIndex_i…`), **reproducible** JSON results, **clear extension points** for new scorers.

Fill in **your** numbers from `results/` when you have them.


## Interview Q and A

**Q: Inverted index?**  
A: Yes — term to **posting lists**; each posting carries **doc id** and scorer-specific payload (tf, positions).

**Q: Why BM25?**  
A: Strong default for **relevance**; uses **IDF** and **length normalization**.

**Q: TF-IDF already?**  
A: Yes — **x=3**; **rare** terms get higher **IDF**.

**Q: Stop words and stemming?**  
A: In **preprocessor**; NLTK stop list + **Porter** stem.

**Q: TAAT vs DAAT in the filename?**  
A: **No** — **runtime** query algorithm.

**Q: Semantic vs lexical?**  
A: **Lexical** = inverted index + BM25. **Semantic** = embeddings in **Chroma** + similarity.

**Q: Biggest limitation?**  
A: **Single machine**; not a distributed search engine.

**Q: How extend to RAG?**  
A: Retrieve top-k with **BM25** or **embeddings**, then pass chunks to an **LLM** — `SearchEngine` or `LocalIndexer` is the retrieval stage.


## This project vs industry tools

| | This repo | Lucene | Managed vector DB |
|--|-----------|--------|-------------------|
| **Purpose** | Learn + experiment | Library in apps | Hosted scale |
| **Lexical** | Custom inverted index | Lucene segments | Varies |
| **Semantic** | Local Chroma + MiniLM | Often via plugins | Native |

**Sound bite:** *“I implemented core IR ideas in Python for transparency; production stacks use similar ideas with more ops and scale.”*


## Trade-offs

| Want | Knob | Cost |
|------|------|------|
| Fastest queries | Boolean, simpler queries | Weaker relevance |
| Smallest disk | Higher z compression | More CPU |
| Best relevance | BM25 + good text | Slower index build |
| Meaning search | Semantic path | Large model download |


## Honest limitations

- **Not** distributed sharding or replication.
- **Semantic** path needs **GPU** optional; CPU works but slower.
- **Evaluation** is performance; **MAP/nDCG** needs labeled relevance data.
- **Streamlit** is a **demo** UI, not production auth or multi-tenant.


## One-page cheat sheet

```text
create_indexer(x,y,z,optim)  →  SelfIndexer
get_index_identifier(...)   →  SelfIndex_i{x}d{y}c{z}o{...}
indexer.create_index(id, docs)
indexer.load_index(id)
indexer.query(text, mode=TAAT|DAAT, top_k=10)

Semantic: pip install -e ".[ui]" ; streamlit run ui.py
```

**Docs:** `docs/ARCHITECTURE.md`, `SCORING_STRATEGIES.md`, `QUERY_PROCESSING.md`, `EVALUATION.md`, `SEMANTIC_SEARCH.md`.


In [ ]:
# Verify imports (run from repo root or notebooks/ with path below)
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd()) == "notebooks" else "."))
sys.path.insert(0, os.path.join(ROOT, "src"))
from ire_search import create_indexer, get_index_identifier
print("Lexical OK:", get_index_identifier(4, 1, 1, "0"))


In [ ]:
import os, sys, tempfile
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd()) == "notebooks" else "."))
sys.path.insert(0, os.path.join(ROOT, "src"))
os.chdir(tempfile.mkdtemp())
os.makedirs("indices", exist_ok=True)
from ire_search import create_indexer
idx = create_indexer(x=4, y=1, z=1, optim="0")
idx.create_index("DEMO", [{"doc_id": "n1", "title": "IR", "content": "information retrieval bm25"}])
print("Query:", idx.query("information retrieval"))


In [ ]:
# Optional: semantic imports (requires pip install -e ".[semantic]" or ".[ui]")
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd()) == "notebooks" else "."))
sys.path.insert(0, os.path.join(ROOT, "src"))
try:
    from ire_search.integrations.chroma_local import LocalIndexer
    print("Semantic imports OK (LocalIndexer available).")
except ImportError as e:
    print("Skip semantic deps:", e)
